In [1]:
# Install required packages in the current environment (run once, then delete this cell)
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm", "ipywidgets", "joblib"])
print("✓ tqdm, ipywidgets, and joblib installed successfully")


✓ tqdm, ipywidgets, and joblib installed successfully


In [2]:
# NNSE Function-based Implementation for Tyson Model
# Implements a vector-based mutation and permutation algorithm

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import copy
import time

# ============================================================================
# === CONFIGURATION VARIABLES ===
# ============================================================================

# Simulation settings
N_STEPS = 3000  # Number of mutation steps to run
SIGMA = 0.01  # Standard deviation for Gaussian mutations in normalized space
N_Vec = 30  # Number of bins for binning squared differences
MAX_VALUE = 15  # Maximum  value for bin thresholds (logspace goes from 0 to this)
K_INITIAL = 1  # Number of top positions to fill initially (top half)
T_START = 0.0  # Simulation start time
T_END = 500.0  # Simulation end time
N_TIME_POINTS = 501  # Number of time points

# Define bin thresholds using logspace (equally spaced in log space)
# y0, y1, ..., yN_BINS: thresholds for binning
bin_thresholds = np.logspace(np.log10(0.1), np.log10(MAX_VALUE), N_Vec + 1)
bin_thresholds[-1] = 1000.0
  # y0, y1, ..., yN_BINS, basically sets y0 to the value of y1 because function will likely never be zero

# ============================================================================
# === BASE PARAMETERS (p0) ===
# ============================================================================

p0 = {
    "k1_aa_over_CT": 0.015,
    "k2": 0.0,
    "k3_CT": 200.0,
    "k4": 180.0,
    "k4prime": 0.018,
    "k5_minusP": 0.0,
    "k6": 1.0,
    "k7": 0.6,
    "k8_minusP": 100.0,
    "k9": 50.0,
    "CT": 1.0
}

# Parameters to vary
param_names = [
    "k1_aa_over_CT",
    "k3_CT",
    "k4",
    "k4prime",
    "k6",
    "k7"
]

p0_vec = np.array([p0[name] for name in param_names])
n_params = len(param_names)

# Time evaluation array
t_eval = np.linspace(T_START, T_END, N_TIME_POINTS)

# ============================================================================
# === TYSON MODEL DEFINITION ===
# ============================================================================

CT = p0["CT"]

def F_M(M, p):
    """Helper function for M-dependent rate"""
    return p["k4prime"] + p["k4"] * (M / p["CT"])**2

def f_rhs(t, x, p):
    """Right-hand side of the ODE system"""
    # x = [C2, CP, pM, M, Y, YP]
    C2, CP, pM, M, Y, YP = x
    k3 = p["k3_CT"] / p["CT"]
    k1 = p["k1_aa_over_CT"] * p['CT']
    dC2 = p["k6"] * M - p["k8_minusP"] * C2 + p["k9"] * CP
    dCP = -k3 * CP * Y + p["k8_minusP"] * C2 - p["k9"] * CP
    dpM = k3 * CP * Y - pM * F_M(M, p) + p["k5_minusP"] * M
    dM  = pM * F_M(M, p) - p["k5_minusP"] * M - p["k6"] * M
    dY  = k1 - p["k2"] * Y - k3 * CP * Y
    dYP = p["k6"] * M - p["k7"] * YP
    return np.array([dC2, dCP, dpM, dM, dY, dYP])

def simulate_at_params(p_local, t_eval):
    """Simulate the ODE system with given parameters"""
    y0 = np.array([0.9, 0.05, 0.0, 0.005, 0.3, 0.0])
    sol = solve_ivp(lambda tt, xx: f_rhs(tt, xx, p_local), 
                    (t_eval[0], t_eval[-1]), y0,
                    method='BDF', t_eval=t_eval, rtol=1e-6, atol=1e-8)
    if not sol.success:
        raise RuntimeError("Integrator failed: " + sol.message)
    return sol.t, sol.y

def compute_obs(X):
    """Compute observables: YT/CT and M/CT"""
    if X.ndim == 1:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT
    else:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT

# ============================================================================
# === REFERENCE SIMULATION (p0) ===
# ============================================================================

print("Running reference simulation with p0...")
t0, y0 = simulate_at_params(p0, t_eval)
YT0, M0 = compute_obs(y0)
print(f"✓ Reference simulation complete")

# ============================================================================
# === SIM FUNCTION ===
# ============================================================================

def sim(P_vec):
    """
    Simulate with parameter vector P and compute squared difference with p0.
    Returns the squared difference (f(xi) value).
    """
    # Convert parameter vector to dictionary
    p_local = copy.deepcopy(p0)
    for i, name in enumerate(param_names):
        p_local[name] = P_vec[i]
    
    # Run simulation
    try:
        t, y = simulate_at_params(p_local, t_eval)
        YT, M = compute_obs(y)
        
        # Interpolate reference to match time points
        YT0_interp = np.interp(t, t0, YT0)
        M0_interp = np.interp(t, t0, M0)
        
        # Compute squared differences
        diff_YT_sq = (YT - YT0_interp)**2
        diff_M_sq = (M - M0_interp)**2
        
        # Integrate squared differences
        integral_YT_sq = np.trapezoid(diff_YT_sq, t)
        integral_M_sq = np.trapezoid(diff_M_sq, t)
        squared_diff = integral_YT_sq + integral_M_sq
        
        return squared_diff
        
    except Exception as e:
        # If simulation fails, return infinity
        print(f"Warning: Simulation failed: {e}")
        return np.inf

print(f"✓ Configuration complete")
print(f"  Parameters: {param_names}")
print(f"  Number of parameters: {n_params}")
print(f"  Bin thresholds (y0, ..., y{N_Vec}): [{bin_thresholds}]")
print


Running reference simulation with p0...
✓ Reference simulation complete
✓ Configuration complete
  Parameters: ['k1_aa_over_CT', 'k3_CT', 'k4', 'k4prime', 'k6', 'k7']
  Number of parameters: 6
  Bin thresholds (y0, ..., y30): [[1.00000000e-01 1.18177929e-01 1.39660229e-01 1.65047567e-01
 1.95049796e-01 2.30505810e-01 2.72406993e-01 3.21924943e-01
 3.80444231e-01 4.49601113e-01 5.31329285e-01 6.27913945e-01
 7.42055697e-01 8.76946055e-01 1.03635669e+00 1.22474487e+00
 1.44737813e+00 1.71048150e+00 2.02141161e+00 2.38886238e+00
 2.82310809e+00 3.33629067e+00 3.94275923e+00 4.65947120e+00
 5.50646657e+00 6.50742816e+00 7.69034384e+00 9.08828909e+00
 1.07403518e+01 1.26927254e+01 1.00000000e+03]]


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [3]:
# ============================================================================
# === TYSONFUNC: MUTATION AND PERMUTATION FUNCTION ===
# ============================================================================

from multiprocessing import Pool
import multiprocessing as mp

def _mutate_worker(args):
    """Worker function for parallel mutation."""
    i, xi, fxi, p0_vec_val, SIGMA_val, n_params_val, bin_thresholds_val = args
    
    if xi is None or fxi is None:
        return (i, None, None)
    
    u_vec = xi / (2.0 * p0_vec_val)
    u_mutated = (u_vec + np.random.normal(0, SIGMA_val, size=n_params_val)) % 1.0
    xi_prime = 2.0 * p0_vec_val * u_mutated
    fxi_prime = sim(xi_prime)
    
    if fxi_prime > bin_thresholds_val[i]:
        return (i, xi.copy(), fxi)
    else:
        return (i, xi_prime, fxi_prime)

def TysonFunc(X_list, fX_list, pool=None):
    """
    Mutate each parameter vector, evaluate, reject if worse, then permute.
    Also handles filling empty positions: if a swap involved an empty position,
    generates a new point and tries to place it as low as possible.
    
    Args:
        X_list: List of parameter vectors [x0, x1, ..., xn] where each xi is a numpy array (may be None)
        fX_list: List of function values [f(x0), f(x1), ..., f(xn)] (may be None)
    
    Returns:
        v_list: List of parameter vectors after mutation and permutation [v0, v1, ..., vn]
        fv_list: List of function values [f(v0), f(v1), ..., f(vn)]
        fX_prime_list: List of function values after mutation but before permutation [f(x'0), f(x'1), ..., f(x'n)]
        swaps: List of (i, j) tuples indicating which positions were swapped
    """
    n = len(X_list)
    
    # Step 1 & 2: Mutate each position
    if pool is not None:
        # PARALLEL: Use worker pool
        work_items = [
            (i, X_list[i], fX_list[i], p0_vec, SIGMA, n_params, bin_thresholds)
            for i in range(n) if X_list[i] is not None
        ]
        if work_items:
            results = pool.map(_mutate_worker, work_items)
            results_dict = {idx: (x, fx) for idx, x, fx in results}
        else:
            results_dict = {}
        X_prime_list = [results_dict.get(i, (None, None))[0] for i in range(n)]
        fX_prime_list = [results_dict.get(i, (None, None))[1] for i in range(n)]
    else:
        # SERIAL: Original loop
        X_prime_list = []
        fX_prime_list = []
        
        for i in range(n):
            xi = X_list[i]
            fxi = fX_list[i]
            
            # Skip empty positions
            if xi is None or fxi is None:
                X_prime_list.append(None)
                fX_prime_list.append(None)
                continue
            
            # Normalize parameters: u_i = p_i / (2 * p0_i) so each lies in [0,1]
            u_vec = xi / (2.0 * p0_vec)
            
            # Apply Gaussian mutation in u-space
            u_mutated = u_vec + np.random.normal(0, SIGMA, size=n_params)
            
            # Wrap around boundaries [0, 1] with periodic boundary conditions
            u_mutated = u_mutated % 1.0
            
            # Map back to parameter space: p_i = 2 * p0_i * u_i
            xi_prime = 2.0 * p0_vec * u_mutated
            
            # Evaluate mutated parameter: f(x'i) := sim(x'i)
            fxi_prime = sim(xi_prime)
            
            # Reject x'i if f(x'i) > y_i, otherwise accept
            if fxi_prime > bin_thresholds[i]:
                # Reject: keep original
                X_prime_list.append(xi.copy())
                fX_prime_list.append(fxi)
            else:
                # Accept: use mutated
                X_prime_list.append(xi_prime)
                fX_prime_list.append(fxi_prime)
    
    # Step 3: Permutation step
    # For each position i from n-1 down to 1 (0-indexed: n-1, n-2, ..., 1)
    # If f(x'_i) <= y_{i-1}, swap position i with position i-1

    v_list = X_prime_list.copy()
    fv_list = fX_prime_list.copy()

    # Track which positions were empty BEFORE swaps
    empty_before_swaps = set(i for i in range(n) if v_list[i] is None or fv_list[i] is None)

    # Track swaps
    swaps = []  # List of (i, j) tuples for swaps

    for i in range(n-1, 0, -1):  # i from n-1 down to 1
        # Skip if both positions are empty
        if fv_list[i] is None and fv_list[i-1] is None:
            continue
        
        # If f(x'_i) <= y_{i-1} and position i is not empty, swap
        if fv_list[i] is not None and fv_list[i] <= bin_thresholds[i-1]:
            # Swap position i with position i-1
            if v_list[i] is not None and v_list[i-1] is not None:
                v_list[i], v_list[i-1] = v_list[i-1].copy(), v_list[i].copy()
            else:
                v_list[i], v_list[i-1] = v_list[i-1], v_list[i]
            fv_list[i], fv_list[i-1] = fv_list[i-1], fv_list[i]
            swaps.append((i, i-1))  # Record the swap

    # Step 4: Fill positions that became empty due to swaps
    # Find positions that are empty AFTER swaps but were NOT empty BEFORE swaps
    empty_after_swaps = set(i for i in range(n) if v_list[i] is None or fv_list[i] is None)
    newly_empty_positions = empty_after_swaps - empty_before_swaps

    # For each newly empty position, generate a new point and place it as low as possible
    for empty_pos in newly_empty_positions:
        # Generate a new random point
        max_attempts = 1000
        for attempt in range(max_attempts):
            # Generate random point in normalized u-space [0, 1]
            u_random = np.random.uniform(0, 1, size=n_params)
            # Map to parameter space: p_i = 2 * p0_i * u_i
            x_new = 2.0 * p0_vec * u_random
            fx_new = sim(x_new)
            
            # Try to place it in the best (lowest index) position it can fit
            placed = False
            for pos in range(n):
                if fx_new <= bin_thresholds[pos]:
                    # Can place it at position pos
                    # If position is empty, place it
                    if v_list[pos] is None or fv_list[pos] is None:
                        v_list[pos] = x_new
                        fv_list[pos] = fx_new
                        placed = True
                        break
                    # If position has a worse value, replace it
                    elif fv_list[pos] is not None and fx_new < fv_list[pos]:
                        v_list[pos] = x_new
                        fv_list[pos] = fx_new
                        placed = True
                        break
            
            if placed:
                break
            # If we couldn't place it anywhere, try again with a new random point
    
    return v_list, fv_list, fX_prime_list, swaps

n_cores = mp.cpu_count()
print(f"✓ TysonFunc defined (parallel & serial modes)")
print(f"  Detected {n_cores} CPU cores available for parallel mode")


✓ TysonFunc defined (parallel & serial modes)
  Detected 12 CPU cores available for parallel mode


In [ ]:
# ============================================================================
# === SIMULATION LOOP ===
# ============================================================================
# Note: fill_vacancies logic is now integrated into TysonFunc
import time
from multiprocessing import Pool
import multiprocessing as mp

try:
    from tqdm.notebook import tqdm
except ImportError:
    # Fallback to text-based progress bar if ipywidgets not available
    from tqdm import tqdm as tqdm_text
    tqdm = tqdm_text
    print("Note: Using text-based progress bar. For better display, install ipywidgets.")

# Helper function to format f values and y thresholds, handling None
def format_f_vals(fX_list, n):
    """Format f values and y thresholds for printing, handling None values.
    Returns two strings: (f_vals_str, y_vals_str)
    Always shows all f(xi) and y_i values.
    """
    indices = list(range(n))
    f_vals = [fX_list[i] if i < len(fX_list) else None for i in indices]
    f_vals_str = " ".join([f"{fx:<12.3e}" if fx is not None else f"{'---':<12}" for fx in f_vals])
    y_vals_str = " ".join([f"{bin_thresholds[i]:<12.3e}" for i in indices])
    return f_vals_str, y_vals_str

n = N_Vec  # Number of parameter vectors to maintain

print(f"Starting simulation with n={n} parameter vectors for {N_STEPS} steps...")

# Create persistent worker pool for parallel mutations
worker_pool = Pool(processes=mp.cpu_count())
print(f"✓ Created worker pool with {mp.cpu_count()} processes")

# Initialize: Only fill worst K positions (highest indices), leave best positions empty (None)
K = K_INITIAL
X_list = [None] * n
fX_list = [None] * n


# Generate K random points in the worst positions (highest indices, highest thresholds)
# Fill positions from n-K to n-1 (e.g., if n=50, K=25, fill positions 25-49)
# SPECIAL: Resample if initial fill would land in the last 2 worst bins (stuck risk)
for idx in range(n-K, n):
    max_init_attempts = 100
    for attempt in range(max_init_attempts):
        # Generate random point in normalized u-space [0, 1]
        u_random = np.random.uniform(0, 1, size=n_params)
        # Map to parameter space: p_i = 2 * p0_i * u_i
        xi = 2.0 * p0_vec * u_random
        fxi = sim(xi)
        
        # Check if this would fit in a bin that's not in the worst 2 positions
        # Find where this value would naturally go
        best_fit_pos = n - 1  # Default to worst position
        for pos in range(n):
            if fxi <= bin_thresholds[pos]:
                best_fit_pos = pos
                break
        
        # If it would land in positions n-2 or n-1 (worst 2 bins) AND we're initializing, resample
        # This prevents getting stuck with terrible initial values
        if best_fit_pos < n - 2 or attempt == max_init_attempts - 1:
            # Accept this value (either it's good, or we've tried enough)
            X_list[idx] = xi
            fX_list[idx] = fxi
            break

# Don't sort during initialization - keep filled positions in their original positions (worst positions)
# The sorting will happen naturally through the mutation and swapping process
filled_count = sum(1 for x in X_list if x is not None)
print(f"\nFilled: {filled_count} positions")

# Print y thresholds row once at the top (in italics)
_, y_vals_str = format_f_vals(fX_list, n)  # Get y_vals_str format

print(f"\n✓ Initialization complete ({filled_count} positions filled, {n-filled_count} empty)")

# Storage for tracking
all_X = [[x.copy() if x is not None else None for x in X_list]]
all_fX = [[fx if fx is not None else None for fx in fX_list]]
all_swaps = [[]]  # Store swaps for each step (empty for initial state)

# Progress tracking
print_interval = max(1, int(N_STEPS // 50)) 

# Timing helpers
def format_duration(seconds: float) -> str:
    if seconds >= 3600:
        return f"{seconds/3600:.2f} hr"
    if seconds >= 60:
        return f"{seconds/60:.2f} min"
    return f"{seconds:.2f} sec"

# Progress bar shown above the table (Jupyter widget version)
pbar = tqdm(total=N_STEPS, desc="Progress", unit="step", leave=True)

# Create table header
# Reordered: Step, #swap, #empty, then all f(xi) values, then time
header_cols = [f"f(x{i})" for i in range(n)]
header = f"{'Step':<6} {'#swap':<6} {'#empty':<7} " + " ".join([f"{col:<12}" for col in header_cols]) + f" {'time':<6}"
separator_len = 6 + 1 + 6 + 1 + 7 + 1 + n * 13 + 1 + 6  # Step + #swap + #empty + n columns + time
tqdm.write(f"\n{header}")
# Print y thresholds row
y_header_cols = [f"y{i}" for i in range(n)]
y_header = f"{'':<6} {'':<6} {'':<7} " + " ".join([f"{col:<12}" for col in y_header_cols]) + f" {'':<6}"
tqdm.write(y_header)

# Print y thresholds row once at the top (in italics)
_, y_vals_str = format_f_vals(fX_list, n)  # Get y_vals_str format
tqdm.write(f"\033[3m{'':<6} {'':<6} {'':<7}  {y_vals_str} {'':<6}\033[0m")  # Italic


# Print initial state
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
empty_count = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_list[i] is None)
tqdm.write(f"{'Init':<6} {'0':<6} {empty_count:<7} {f_vals_str} {'0.00':<6}")

BURN_IN = int(0.5*N_STEPS)  # Wait for population to stabilize
swap_count = np.zeros(N_Vec + 1)  # Track swaps at each position
total_opportunities = np.zeros(N_Vec + 1)  # Track total chances to swap
is_burned_in = False

# NEW: Store volume ratio history for convergence analysis
volume_ratio_history = []  # List of (step, volume_ratios) tuples
history_interval = 100  # Save every 100 steps after burn-in

last_print_time = time.time()
for step in range(N_STEPS):
    # Store unmutated state (before mutation)
    fX_unmutated = [fx if fx is not None else None for fx in fX_list]
    
    # Apply TysonFunc (now includes fill spots logic) with parallel pool
    X_list, fX_list, fX_mutated, swaps = TysonFunc(X_list, fX_list, pool=worker_pool)
    
    # Track swap statistics (only after burn-in)
    if step >= BURN_IN:
        if not is_burned_in:
            # Reset counters at burn-in point
            swap_count[:] = 0
            total_opportunities[:] = 0
            is_burned_in = True
            tqdm.write(f"\n=== BURN-IN COMPLETE at step {step} - Starting volume estimation ===\n")
        
        # Count swaps and opportunities at each position
        # Convert swap tuples to a set for quick lookup
        swap_set = set(swaps)  # swaps is a list of (i, i-1) tuples
        
        for i in range(1, len(fX_list)):  # positions 1 to n
            # Only count if both positions are filled (not None)
            if fX_list[i] is not None and fX_list[i-1] is not None:
                total_opportunities[i] += 1
                # Check if position i swapped with position i-1
                if (i, i-1) in swap_set:
                    swap_count[i] += 1
        
        # NEW: Store volume ratios at intervals for convergence tracking
        if (step - BURN_IN) % history_interval == 0 and step > BURN_IN:
            current_ratios = np.zeros(N_Vec + 1)
            for i in range(1, len(swap_count)):
                if total_opportunities[i] > 0:
                    current_ratios[i] = swap_count[i] / total_opportunities[i]
            volume_ratio_history.append((step, current_ratios.copy()))
    
    # Store trajectory (handle None values)
    all_X.append([x.copy() if x is not None else None for x in X_list])
    all_fX.append([fx if fx is not None else None for fx in fX_list])
    all_swaps.append(swaps)
    
    # Progress report in table format
    if (step + 1) % print_interval == 0 or step == 0:
        # Calculate time since last print
        current_time = time.time()
        interval_time = current_time - last_print_time
        last_print_time = current_time

        interval_time_str = format_duration(interval_time)

        # Count empty positions in unmutated state
        empty_count_unmut = sum(1 for i in range(len(fX_unmutated)) if fX_unmutated[i] is None)

        # Print unmutated row
        f_vals_str, y_vals_str = format_f_vals(fX_unmutated, n)
        tqdm.write(f"{step+1:<6} {'-':<6} {empty_count_unmut:<7} {f_vals_str} {interval_time_str:<12}")

        # Count empty positions in current state
        empty_count = sum(1 for i in range(len(fX_list)) if fX_list[i] is None)

        # Print mutated row
        f_vals_str, y_vals_str = format_f_vals(fX_mutated, n)
        tqdm.write(f"{'':<6} {len(swaps):<6} {empty_count:<7} {f_vals_str} {interval_time_str:<12}")
                
        # If past burn-in, also print volume estimates
        if is_burned_in and step >= BURN_IN + 100:  # Wait a bit after burn-in
            tqdm.write(f"\n--- Volume Estimation (Step {step + 1}) ---")
            for i in range(1, len(swap_count)):
                if total_opportunities[i] > 0:
                    ratio = swap_count[i] / total_opportunities[i]
                    tqdm.write(f"  Position {i}: swap_freq = {ratio:.4f} ({int(swap_count[i])}/{int(total_opportunities[i])})")
        
        # Update progress bar to current position (absolute, not incremental)
        pbar.n = step + 1
        pbar.refresh()

# Close progress bar before final output
pbar.close()

# After the loop, compute final volume estimates
if is_burned_in:
    tqdm.write("\n=== Final Volume Ratios ===")
    volume_ratios = np.zeros(N_Vec + 1)
    for i in range(1, len(swap_count)):
        if total_opportunities[i] > 0:
            volume_ratios[i] = swap_count[i] / total_opportunities[i]
            tqdm.write(f"V[{i-1}]/V[{i}] = {volume_ratios[i]:.6f}")
    
    # NEW: Show convergence of volume ratios over time
    if len(volume_ratio_history) > 0:
        tqdm.write("\n=== Volume Ratio Convergence Over Time ===")
        tqdm.write(f"Tracked at {len(volume_ratio_history)} timepoints")
        
        # Show first few positions' convergence
        for i in range(1, min(6, N_Vec + 1)):
            ratios_over_time = [item[1][i] for item in volume_ratio_history]
            steps = [item[0] for item in volume_ratio_history]
            if len(ratios_over_time) > 0:
                tqdm.write(f"\nPosition {i}:")
                tqdm.write(f"  Steps: {steps}")
                tqdm.write(f"  Ratios: {[f'{r:.4f}' for r in ratios_over_time]}")
                if len(ratios_over_time) > 1:
                    variance = np.var(ratios_over_time[-5:]) if len(ratios_over_time) >= 5 else np.var(ratios_over_time)
                    tqdm.write(f"  Recent variance: {variance:.6f}")

# Close the worker pool
worker_pool.close()
worker_pool.join()

tqdm.write(f"\n✓ Simulation complete!")
# Print final state in table format
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
empty_count = sum(1 for i in range(len(fX_list)) if fX_list[i] is None)
current_time = time.time()
final_interval_time = current_time - last_print_time
final_interval_time_str = format_duration(final_interval_time)

tqdm.write(f"{'Final':<6} {'0':<6} {empty_count:<7} {f_vals_str} {final_interval_time_str:<12}")

# Convert to numpy arrays for easier analysis
# Replace None with np.nan for fX (scalar values)
all_fX_clean = [[fx if fx is not None else np.nan for fx in step_fX] for step_fX in all_fX]
all_fX_array = np.array(all_fX_clean)  # Shape: (N_STEPS+1, n)

# For X (parameter vectors), replace None with NaN-filled arrays
all_X_clean = []
for step_X in all_X:
    step_X_clean = []
    for x in step_X:
        if x is not None:
            step_X_clean.append(x.copy())
        else:
            # Use NaN-filled array as placeholder for empty positions
            step_X_clean.append(np.full(n_params, np.nan))
    all_X_clean.append(step_X_clean)
all_X_array = np.array(all_X_clean)  # Shape: (N_STEPS+1, n, n_params)






Starting simulation with n=30 parameter vectors for 3000 steps...


✓ Created worker pool with 12 processes

Filled: 1 positions

✓ Initialization complete (1 positions filled, 29 empty)


Progress:   0%|          | 0/3000 [00:00<?, ?step/s]


Step   #swap  #empty  f(x0)        f(x1)        f(x2)        f(x3)        f(x4)        f(x5)        f(x6)        f(x7)        f(x8)        f(x9)        f(x10)       f(x11)       f(x12)       f(x13)       f(x14)       f(x15)       f(x16)       f(x17)       f(x18)       f(x19)       f(x20)       f(x21)       f(x22)       f(x23)       f(x24)       f(x25)       f(x26)       f(x27)       f(x28)       f(x29)       time  
                      y0           y1           y2           y3           y4           y5           y6           y7           y8           y9           y10          y11          y12          y13          y14          y15          y16          y17          y18          y19          y20          y21          y22          y23          y24          y25          y26          y27          y28          y29                
                       1.000e-01    1.182e-01    1.397e-01    1.650e-01    1.950e-01    2.305e-01    2.724e-01    3.219e-01    3.804e-01    4.496e-01    5.313e-0

In [ ]:
# ============================================================================
# === VISUALIZATION: PCA TRAJECTORY AND DISTRIBUTIONS ===
# ============================================================================

from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

print("Preparing data for visualization...")

# Convert to numpy arrays, handling None values
# For X: use np.nan where None, for fX: use np.nan where None
all_X_array = np.array([[x if x is not None else np.nan * np.ones(n_params) for x in step_X] for step_X in all_X])
all_fX_array = np.array([[fx if fx is not None else np.nan for fx in step_fX] for step_fX in all_fX])

# Extract the best vector at each step
# Since vectors are sorted by f value, find the first non-NaN value (best available)
# This handles the case where x0 might be empty for many steps
trajectory = []
squared_diffs = []
step_indices = []  # Keep track of which steps had valid data

for step_idx in range(len(all_X_array)):
    # Find the first non-NaN (best) vector at this step
    for pos_idx in range(n):
        if not np.isnan(all_fX_array[step_idx, pos_idx]):
            trajectory.append(all_X_array[step_idx, pos_idx])
            squared_diffs.append(all_fX_array[step_idx, pos_idx])
            step_indices.append(step_idx)
            break  # Found the best available vector, move to next step

trajectory = np.array(trajectory)
squared_diffs = np.array(squared_diffs)
step_indices = np.array(step_indices)

print(f"  Total steps: {len(all_X_array)}")
print(f"  Steps with valid data: {len(trajectory)}")
print(f"  First valid step: {step_indices[0] if len(step_indices) > 0 else 'None'}")
print(f"  Best f value trajectory: {len(squared_diffs)} points")

# ============================================================================
# === PCA TRAJECTORY PLOT ===
# ============================================================================

print("\nComputing PCA for trajectory visualization...")

# Normalize trajectory for PCA (use standardized parameters)
trajectory_normalized = (trajectory - trajectory.mean(axis=0)) / (trajectory.std(axis=0) + 1e-10)

# Compute PCA
pca = PCA(n_components=min(3, n_params))
trajectory_pca = pca.fit_transform(trajectory_normalized)

print(f"✓ PCA complete")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Simulate with first and last parameters for YT/CT comparison
print("Simulating with first and last parameters...")
P_first = trajectory[0]  # Best parameter vector at step 0
P_last = trajectory[-1]  # Best parameter vector at final step

# Convert to parameter dictionaries
p_first = copy.deepcopy(p0)
p_last = copy.deepcopy(p0)
for i, name in enumerate(param_names):
    p_first[name] = P_first[i]
    p_last[name] = P_last[i]

# Simulate
t_first, y_first = simulate_at_params(p_first, t_eval)
YT_first, M_first = compute_obs(y_first)
t_last, y_last = simulate_at_params(p_last, t_eval)
YT_last, M_last = compute_obs(y_last)

print("✓ Simulations complete")

# Create figure with 3 subplots
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# Plot 2D PCA trajectory
ax1 = fig.add_subplot(gs[0, 0])
scatter = ax1.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], 
                     c=range(len(trajectory_pca)), cmap='viridis', 
                     s=20, alpha=0.6, edgecolors='none')
ax1.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], 'k-', alpha=0.3, linewidth=0.5)
ax1.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], 
           color='red', s=100, marker='o', label='Start', zorder=5)
ax1.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], 
           color='blue', s=100, marker='s', label='End', zorder=5)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
ax1.set_title('NNSE Trajectory in PCA Space (PC1 vs PC2)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Step')

# Second subplot: 3D plot or squared difference over time
if trajectory_pca.shape[1] >= 3:
    from mpl_toolkits.mplot3d import Axes3D
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    scatter2 = ax2.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2],
                         c=range(len(trajectory_pca)), cmap='viridis', s=20, alpha=0.6)
    ax2.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2], 
            'k-', alpha=0.3, linewidth=0.5)
    ax2.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], trajectory_pca[0, 2],
               color='red', s=100, marker='o', label='Start')
    ax2.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], trajectory_pca[-1, 2],
               color='blue', s=100, marker='s', label='End')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=10)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=10)
    ax2.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.2%})', fontsize=10)
    ax2.set_title('NNSE Trajectory in PCA Space (3D)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    plt.colorbar(scatter2, ax=ax2, label='Step')
else:
    # If only 2 components, show squared difference over time
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(range(len(squared_diffs)), squared_diffs, 'b-', linewidth=1, alpha=0.7)
    ax2.set_xlabel('Step', fontsize=12)
    ax2.set_ylabel('Squared Difference', fontsize=12)
    ax2.set_title('Best Squared Difference Over Time', fontsize=14, fontweight='bold')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)

# Third subplot: YT/CT comparison
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(t0, YT0, 'k-', lw=2, label='Reference (p0)', alpha=0.8)
ax3.plot(t_first, YT_first, 'r-', lw=2, label='First parameter', alpha=0.7)
ax3.plot(t_last, YT_last, 'b-', lw=2, label='Last parameter', alpha=0.7)
ax3.set_xlabel('Time (min)', fontsize=12)
ax3.set_ylabel('YT/CT', fontsize=12)
ax3.set_title('YT/CT Comparison', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# === DISTRIBUTION OF SQUARED DIFFERENCES ===
# ============================================================================

print("\nPlotting distribution of squared differences...")

# For distribution, use all squared differences from all vectors at all steps
# This gives a better picture of the full distribution
squared_diffs_all = all_fX_array.flatten()  # All function values from all vectors

# Filter out infinite values
valid_mask = np.isfinite(squared_diffs_all)
squared_diffs_valid = squared_diffs_all[valid_mask]

print(f"  Valid values: {np.sum(valid_mask)}/{len(squared_diffs_all)}")
print(f"  Mean: {np.mean(squared_diffs_valid):.6e}")
print(f"  Median: {np.median(squared_diffs_valid):.6e}")
print(f"  Min: {np.min(squared_diffs_valid):.6e}")
print(f"  Max: {np.max(squared_diffs_valid):.6e}")

# Create distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with linear bins
ax1 = axes[0]
n_bins_hist = 50
# Use linear bins from min to max
counts, bins_hist, patches = ax1.hist(squared_diffs_valid, bins=n_bins_hist, 
                                      edgecolor='black', alpha=0.7, color='steelblue',
                                      density=True)
ax1.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
ax1.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
           linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
ax1.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Squared Differences (Histogram)', 
             fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Kernel density estimate (KDE) with linear scale
ax2 = axes[1]
if len(squared_diffs_valid) > 1:
    # Use linear KDE (not log-transformed)
    kde = gaussian_kde(squared_diffs_valid)
    x_kde = np.linspace(squared_diffs_valid.min(), squared_diffs_valid.max(), 200)
    density = kde(x_kde)
    ax2.plot(x_kde, density, 'b-', linewidth=2, label='KDE')
    ax2.fill_between(x_kde, 0, density, alpha=0.3, color='steelblue')
    ax2.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
    ax2.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
    ax2.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Not enough data for KDE', 
            ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")


Manually running simplified version of seed=3 simulation...

✓ Pool created

Initializing 1 positions...
  Position 29: f = 7.99e+01

✓ Initialized: 1 filled, 29 empty

Running first 10 steps...
  Step 1: 0 swaps, 1 filled positions
    First few f values: ['None', 'None', 'None', 'None', 'None']
  Step 2: 0 swaps, 1 filled positions
  Step 3: 0 swaps, 1 filled positions
  Step 4: 0 swaps, 1 filled positions
  Step 5: 0 swaps, 1 filled positions
  Step 6: 0 swaps, 1 filled positions
  Step 7: 0 swaps, 1 filled positions
  Step 8: 0 swaps, 1 filled positions
  Step 9: 0 swaps, 1 filled positions
  Step 10: 0 swaps, 1 filled positions

✓ Pool closed


In [ ]:
# ============================================================================
# === JACKKNIFE STABILITY DIAGNOSTIC OVER INDEPENDENT FULL RUNS ===
# ============================================================================
# Treat each *full* simulation run (different seed) as one observation.
# We do NOT jackknife over internal steps/states.

import contextlib
import io
import time
import numpy as np
from multiprocessing import Pool
import multiprocessing as mp

from stability_jackknife import run_jackknife


def run_volume_ratio_simulation(seed: int, *, verbose: bool = False) -> np.ndarray:
    """Run one full simulation with a given seed and return final volume ratios.

    Returns
    -------
    ratios : np.ndarray
        Shape (N_Vec,), corresponding to V[i-1]/V[i] for i=1..N_Vec.
        (Index 0 is omitted because it's unused in the notebook code.)

    Notes
    -----
    - Uses the existing simulation logic (TysonFunc + swap counting) unchanged.
    - Re-initializes all per-run state so runs are independent.
    """
    np.random.seed(int(seed))

    def _run() -> np.ndarray:
        n = N_Vec

        # Create worker pool for this jackknife run (same as main loop)
        jk_pool = Pool(processes=mp.cpu_count())

        # Initialize population (same logic as the main simulation cell)
        K = K_INITIAL
        X_list = [None] * n
        fX_list = [None] * n

        # SPECIAL: Resample if initial fill would land in the last 2 worst bins (stuck risk)
        for idx in range(n-K, n):
            max_init_attempts = 100
            for attempt in range(max_init_attempts):
                # Generate random point in normalized u-space [0, 1]
                u_random = np.random.uniform(0, 1, size=n_params)
                # Map to parameter space: p_i = 2 * p0_i * u_i
                xi = 2.0 * p0_vec * u_random
                fxi = sim(xi)
                
                # Check if this would fit in a bin that's not in the worst 2 positions
                # Find where this value would naturally go
                best_fit_pos = n - 1  # Default to worst position
                for pos in range(n):
                    if fxi <= bin_thresholds[pos]:
                        best_fit_pos = pos
                        break
                
                # If it would land in positions n-2 or n-1 (worst 2 bins) AND we're initializing, resample
                # This prevents getting stuck with terrible initial values
                if best_fit_pos < n - 2 or attempt == max_init_attempts - 1:
                    # Accept this value (either it's good, or we've tried enough)
                    X_list[idx] = xi
                    fX_list[idx] = fxi
                    break

        BURN_IN = int(0.5 * N_STEPS)
        swap_count = np.zeros(N_Vec + 1)
        total_opportunities = np.zeros(N_Vec + 1)
        is_burned_in = False

        for step in range(N_STEPS):
            # Use parallel mode with worker pool (same as main loop)
            X_list, fX_list, _fX_mutated, swaps = TysonFunc(X_list, fX_list, pool=jk_pool)

            if step >= BURN_IN:
                if not is_burned_in:
                    swap_count[:] = 0
                    total_opportunities[:] = 0
                    is_burned_in = True

                swap_set = set(swaps)
                for i in range(1, len(fX_list)):
                    if fX_list[i] is not None and fX_list[i - 1] is not None:
                        total_opportunities[i] += 1
                        if (i, i - 1) in swap_set:
                            swap_count[i] += 1

        if not is_burned_in:
            raise RuntimeError("Burn-in was never reached; increase N_STEPS.")

        volume_ratios = np.zeros(N_Vec + 1)
        for i in range(1, len(swap_count)):
            if total_opportunities[i] > 0:
                volume_ratios[i] = swap_count[i] / total_opportunities[i]
            else:
                volume_ratios[i] = np.nan

        # Close worker pool for this run
        jk_pool.close()
        jk_pool.join()

        return volume_ratios[1:].copy()

    if verbose:
        return _run()

    # Silence prints from lower-level code during repeated runs
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        return _run()


# --- Configure jackknife runs ---
N_RUNS = 20
SEED0 = 0

# Run jackknife simulations SEQUENTIALLY (one at a time)
# Each run uses parallel workers internally (same as main loop)
print(f"Running {N_RUNS} jackknife simulations sequentially...")
print(f"Each run uses {mp.cpu_count()} cores for parallel speedup\n")

start_all = time.time()
seeds = []
runs = []

for i in range(N_RUNS):
    seed = SEED0 + i
    print(f"  Starting run {i+1}/{N_RUNS} (seed={seed})...")
    run_start = time.time()
    
    ratios = run_volume_ratio_simulation(seed, verbose=False)
    
    run_time = time.time() - run_start
    elapsed_total = time.time() - start_all
    remaining = N_RUNS - (i + 1)
    avg_per_run = elapsed_total / (i + 1)
    eta = remaining * avg_per_run
    
    print(f"  ✓ Run {i+1}/{N_RUNS} complete in {run_time/60:.1f} min | "
          f"Total elapsed: {elapsed_total/60:.1f} min | ETA: {eta/60:.1f} min\n")
    
    seeds.append(seed)
    runs.append(ratios)

runs = np.array(runs)

# Compute jackknife statistics manually
from collections import namedtuple
JKResult = namedtuple('JKResult', ['names', 'theta_hat', 'bias', 'se', 'rel_bias', 'rel_se', 
                                     'ok_rel_bias', 'ok_rel_se', 'rel_bias_threshold', 'rel_se_threshold'])

names = [f"V[{i-1}]/V[{i}]" for i in range(1, N_Vec + 1)]
theta_hat = np.nanmean(runs, axis=0)
n_runs = len(runs)

# Jackknife estimates
jackknife_estimates = np.zeros((n_runs, runs.shape[1]))
for i in range(n_runs):
    leave_one_out = np.delete(runs, i, axis=0)
    jackknife_estimates[i] = np.nanmean(leave_one_out, axis=0)

bias = (n_runs - 1) * (np.nanmean(jackknife_estimates, axis=0) - theta_hat)
se = np.sqrt((n_runs - 1) * np.nanmean((jackknife_estimates - theta_hat)**2, axis=0))

rel_bias = np.abs(bias / theta_hat)
rel_se = se / np.abs(theta_hat)

ok_rel_bias = rel_bias < 0.05
ok_rel_se = rel_se < 0.05

jk = JKResult(names, theta_hat, bias, se, rel_bias, rel_se, ok_rel_bias, ok_rel_se, 0.05, 0.05)

wall_time = time.time() - start_all

print(f"\nJackknife over {len(seeds)} independent full runs")
print(f"Seeds: {seeds[:5]}...{seeds[-5:]}")
print(f"Timing: total={wall_time:.1f}s ({wall_time/60:.1f} min)")

# Compact report: show first few ratios
k_show = min(8, len(jk.names))
for j in range(k_show):
    name = jk.names[j]
    th = jk.theta_hat[j]
    bias = jk.bias[j]
    se = jk.se[j]
    print(
        f"{name:<12}  theta_hat={th: .6g}  bias={bias: .3g}  se={se: .3g}  "
        f"rel_bias={jk.rel_bias[j]:.3g}  rel_se={jk.rel_se[j]:.3g}  "
        f"OK(bias)={bool(jk.ok_rel_bias[j])}  OK(se)={bool(jk.ok_rel_se[j])}"
    )

n_ok_bias = int(np.sum(jk.ok_rel_bias))
n_ok_se = int(np.sum(jk.ok_rel_se))
print(
    f"\nSummary: OK rel-bias for {n_ok_bias}/{len(jk.names)} stats, "
    f"OK rel-SE for {n_ok_se}/{len(jk.names)} stats "
    f"(thresholds: {jk.rel_bias_threshold}, {jk.rel_se_threshold})"
)

Running 20 jackknife simulations sequentially...
Each run uses 12 cores for parallel speedup

  Starting run 1/20 (seed=0)...
  ✓ Run 1/20 complete in 88.3 min | Total elapsed: 88.3 min | ETA: 1677.3 min

  Starting run 2/20 (seed=1)...
  ✓ Run 2/20 complete in 63.8 min | Total elapsed: 152.1 min | ETA: 1368.8 min

  Starting run 3/20 (seed=2)...
  ✓ Run 3/20 complete in 87.6 min | Total elapsed: 239.7 min | ETA: 1358.5 min

  Starting run 4/20 (seed=3)...
  ✓ Run 4/20 complete in 2.4 min | Total elapsed: 242.1 min | ETA: 968.5 min

  Starting run 5/20 (seed=4)...


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# === JACKKNIFE RESULTS VISUALIZATION ===
# ============================================================================

import matplotlib.pyplot as plt

print("Visualizing jackknife results...")
print(f"Data: {len(runs)} runs × {runs.shape[1]} volume ratios\n")

# Create figure with multiple subplots
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Plot all volume ratios with error bars
ax1 = fig.add_subplot(gs[0, :])
positions = np.arange(len(jk.theta_hat))
ax1.errorbar(positions, jk.theta_hat, yerr=jk.se, fmt='o-', capsize=5, 
             markersize=6, linewidth=2, color='steelblue', label='Mean ± SE')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel('Position Index', fontsize=12, fontweight='bold')
ax1.set_ylabel('Volume Ratio V[i-1]/V[i]', fontsize=12, fontweight='bold')
ax1.set_title(f'Volume Ratios Across All Positions (N={N_RUNS} runs)', 
              fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)
ax1.set_xlim(-0.5, len(positions) - 0.5)

# 2. Distribution of ratios for selected positions (first 6)
ax2 = fig.add_subplot(gs[1, 0])
n_show = min(6, runs.shape[1])
colors = plt.cm.viridis(np.linspace(0, 1, n_show))
for i in range(n_show):
    ratios_i = runs[:, i]
    valid_ratios = ratios_i[~np.isnan(ratios_i)]
    if len(valid_ratios) > 0:
        ax2.hist(valid_ratios, bins=10, alpha=0.6, label=f'V[{i}]/V[{i+1}]', 
                color=colors[i], edgecolor='black')
ax2.set_xlabel('Volume Ratio', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count', fontsize=12, fontweight='bold')
ax2.set_title(f'Distribution of First {n_show} Ratios Across Runs', 
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# 3. Relative standard error by position
ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(positions, jk.rel_se, color='coral', edgecolor='black', alpha=0.7)
ax3.axhline(0.05, color='red', linestyle='--', linewidth=2, label='Threshold (0.05)')
ax3.set_xlabel('Position Index', fontsize=12, fontweight='bold')
ax3.set_ylabel('Relative Standard Error', fontsize=12, fontweight='bold')
ax3.set_title('Uncertainty by Position (lower is better)', 
              fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')
ax3.set_xlim(-0.5, len(positions) - 0.5)

# 4. Run-to-run variability (show all runs for selected positions)
ax4 = fig.add_subplot(gs[2, 0])
n_show_runs = min(8, runs.shape[1])
for i in range(n_show_runs):
    ratios_i = runs[:, i]
    valid_mask = ~np.isnan(ratios_i)
    run_indices = np.arange(len(ratios_i))[valid_mask]
    valid_ratios = ratios_i[valid_mask]
    if len(valid_ratios) > 0:
        ax4.plot(run_indices, valid_ratios, 'o-', markersize=6, 
                label=f'V[{i}]/V[{i+1}]', alpha=0.7)
ax4.set_xlabel('Run Index', fontsize=12, fontweight='bold')
ax4.set_ylabel('Volume Ratio', fontsize=12, fontweight='bold')
ax4.set_title(f'Run-to-Run Variability (First {n_show_runs} Positions)', 
              fontsize=14, fontweight='bold')
ax4.legend(fontsize=9, ncol=2)
ax4.grid(True, alpha=0.3)

# 5. Quality check: show which positions passed/failed
ax5 = fig.add_subplot(gs[2, 1])
quality_scores = (jk.ok_rel_bias.astype(int) + jk.ok_rel_se.astype(int)) / 2
colors_qual = ['red' if q < 0.5 else 'orange' if q < 1 else 'green' for q in quality_scores]
ax5.bar(positions, quality_scores, color=colors_qual, edgecolor='black', alpha=0.7)
ax5.set_xlabel('Position Index', fontsize=12, fontweight='bold')
ax5.set_ylabel('Quality Score (0=fail, 0.5=partial, 1=pass)', fontsize=12, fontweight='bold')
ax5.set_title('Jackknife Quality by Position', fontsize=14, fontweight='bold')
ax5.set_ylim(-0.1, 1.1)
ax5.grid(True, alpha=0.3, axis='y')
ax5.set_xlim(-0.5, len(positions) - 0.5)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"JACKKNIFE SUMMARY STATISTICS")
print(f"{'='*60}")
print(f"Total runs: {N_RUNS}")
print(f"Positions analyzed: {len(jk.names)}")
print(f"Passed bias check: {np.sum(jk.ok_rel_bias)}/{len(jk.names)}")
print(f"Passed SE check: {np.sum(jk.ok_rel_se)}/{len(jk.names)}")
print(f"\nMean volume ratio: {np.nanmean(jk.theta_hat):.4f}")
print(f"Median SE: {np.nanmedian(jk.se):.4f}")
print(f"Mean relative SE: {np.nanmean(jk.rel_se):.4f}")
print(f"\nPositions with highest uncertainty (top 5 by rel_se):")
worst_se_idx = np.argsort(jk.rel_se)[-5:][::-1]
for idx in worst_se_idx:
    print(f"  {jk.names[idx]}: rel_se = {jk.rel_se[idx]:.4f}, theta_hat = {jk.theta_hat[idx]:.4f}")

print(f"\n✓ Jackknife visualization complete")


Run 4 volume ratios:
[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan]
